## **Mapping Topik BERTopic -> 15 Aspek (rujuk FiQA)**

In [17]:
from pathlib import Path
import pandas as pd

sent = pd.read_csv("data/processed/sentences_with_topic.csv", keep_default_na=False)
info = pd.read_csv("data/processed/topic_info.csv")
print(f"Kalimat: {len(sent):,} | Topik: {len(info)}")

Kalimat: 63,627 | Topik: 43


In [18]:
sent.info()

<class 'pandas.DataFrame'>
RangeIndex: 63627 entries, 0 to 63626
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id_artikel  63627 non-null  int64
 1   id_kalimat  63627 non-null  str  
 2   bank        63627 non-null  str  
 3   timestamp   63627 non-null  str  
 4   kalimat     63627 non-null  str  
 5   n_kata      63627 non-null  int64
 6   topic       63627 non-null  int64
dtypes: int64(3), str(4)
memory usage: 13.1 MB


## 15 aspek final + pemetaan tiap topik

`ASPEK` mendefinisikan 15 aspek (nama + kategori FiQA). `TOPIC2ASPEK` memetakan tiap topik ke salah satu aspek, atau ke `Lainnya`. Edit bebas setelah membaca kalimat representatif.

In [19]:
# 15 ASPEK FINAL: nama -> kategori FiQA
ASPEK = {
    # CORPORATE (6)
    "Kinerja Keuangan & Laba":            "Corporate",
    "Kredit & Kualitas Aset":             "Corporate",
    "Dividen & Aksi Korporasi":           "Corporate",
    "Manajemen & Kepemimpinan":           "Corporate",
    "Layanan & Produk":                   "Corporate",
    "Strategi & Ekspansi Bisnis":         "Corporate",

    # STOCK (4)
    "Pergerakan Saham Bank":              "Stock",
    "Aksi Investor Asing":                "Stock",
    "Saham Spesifik Non-Bank":            "Stock",
    "Valuasi & Kapitalisasi":             "Stock",

    # MARKET (3)
    "Pergerakan IHSG & Indeks":           "Market",
    "Sektor & Transaksi Pasar":           "Market",
    "Bursa Global":                       "Market",

    # ECONOMY (2)
    "Kebijakan Moneter & Nilai Tukar":    "Economy",
    "Makroekonomi & Faktor Global":       "Economy",
}
assert len(ASPEK) == 15, f"harus 15 aspek, sekarang {len(ASPEK)}"


# pemetaan tiap topik -> nama aspek (atau 'Lainnya')
TOPIC2ASPEK = {
    -1: "Lainnya",

    # Corporate
    0:  "Kredit & Kualitas Aset",
    2:  "Strategi & Ekspansi Bisnis",
    7:  "Dividen & Aksi Korporasi",
    8:  "Strategi & Ekspansi Bisnis",
    11: "Kinerja Keuangan & Laba",
    16: "Manajemen & Kepemimpinan",
    21: "Layanan & Produk",
    22: "Layanan & Produk",
    23: "Dividen & Aksi Korporasi",
    24: "Kredit & Kualitas Aset",
    25: "Layanan & Produk",
    27: "Dividen & Aksi Korporasi",
    32: "Kinerja Keuangan & Laba",
    37: "Kredit & Kualitas Aset",

    # Stock
    9:  "Aksi Investor Asing",
    13: "Pergerakan Saham Bank",
    19: "Aksi Investor Asing",
    20: "Pergerakan Saham Bank",
    29: "Pergerakan Saham Bank",
    31: "Saham Spesifik Non-Bank",
    33: "Aksi Investor Asing",
    35: "Saham Spesifik Non-Bank",
    38: "Valuasi & Kapitalisasi",
    39: "Pergerakan Saham Bank",
    41: "Pergerakan Saham Bank",

    # Market
    1:  "Pergerakan IHSG & Indeks",
    3:  "Bursa Global",
    4:  "Aksi Investor Asing",
    5:  "Sektor & Transaksi Pasar",
    6:  "Sektor & Transaksi Pasar",
    10: "Pergerakan IHSG & Indeks",
    12: "Pergerakan IHSG & Indeks",
    28: "Bursa Global",
    34: "Bursa Global",
    36: "Bursa Global",

    # Economy
    14: "Kebijakan Moneter & Nilai Tukar",
    17: "Kebijakan Moneter & Nilai Tukar",
    18: "Makroekonomi & Faktor Global",
    26: "Kebijakan Moneter & Nilai Tukar",
    30: "Makroekonomi & Faktor Global",
    40: "Makroekonomi & Faktor Global",

    # Lainnya
    15: "Lainnya",
}


# cek: semua topik tercakup
topik_data = set(int(t) for t in info["Topic"])
belum = sorted(topik_data - set(TOPIC2ASPEK))
tidak_ada = sorted(set(TOPIC2ASPEK) - topik_data)

print("Topik belum dipetakan:", belum if belum else "tidak ada (semua tercakup)")
print("Mapping untuk topik yang tidak ada:", tidak_ada if tidak_ada else "tidak ada")
print(f"Jumlah aspek: {len(ASPEK)} (+ Lainnya)")

Topik belum dipetakan: tidak ada (semua tercakup)
Mapping untuk topik yang tidak ada: tidak ada
Jumlah aspek: 15 (+ Lainnya)


## Tabel mapping per topik (untuk dicek manual)

In [20]:
def aspek_of(t):
    return TOPIC2ASPEK.get(int(t), "Lainnya")
def kategori_of(a):
    return ASPEK.get(a, "Lainnya")

info["nama_aspek"]   = info["Topic"].map(aspek_of)
info["kategori_fiqa"] = info["nama_aspek"].map(kategori_of)

mapping = info[["Topic", "Count", "kategori_fiqa", "nama_aspek", "Keywords",
                "Representative_1", "Representative_2", "Representative_3"]].copy()
mapping = mapping.sort_values(["kategori_fiqa", "nama_aspek", "Topic"]).reset_index(drop=True)
mapping

,Topic,Count,kategori_fiqa,nama_aspek,Keywords,Representative_1,Representative_2,Representative_3
0,7,1355,Corporate,Dividen & Aksi Korporasi,"2026, 2025, tanggal, dividen, rapat, dijadwalk...",Cum date dividen untuk pasar reguler dan negos...,Keputusan tersebut telah memperoleh persetujua...,Perseroan menetapkan tanggal cum dividen di pa...
1,23,605,Corporate,Dividen & Aksi Korporasi,"per saham, per, dividen, saham, tahun buku, bu...",PT Bank Mandiri (Persero) Tbk. (BMRI) sepakat ...,"Bila ditelusuri, AHI tercatat membagikan divid...","Sebagai informasi, BBRI membagikan dividen int..."
2,27,536,Corporate,Dividen & Aksi Korporasi,"buyback, pembelian saham, pembelian, saham, ma...",Rapat Umum Pemegang Saham (RPUS) PT Bank Negar...,PT Bank Negara Indonesia (Persero) Tbk. (BBNI)...,Periode pelaksanaan buyback dilakukan dalam wa...
3,11,1121,Corporate,Kinerja Keuangan & Laba,"laba, pendapatan, sebesar, laba bersih, mening...","Pada semester I-2026, MKPI membukukan pendapat...","Laba bersih tercatat Rp14,33 triliun, meningka...",Laba bersih perseroan melonjak 739% secara tah...
4,32,349,Corporate,Kinerja Keuangan & Laba,"bri, tumbuh, rp1, total, sebesar, penyaluran, ...","""Dari sisi intermediasi, penyaluran kredit BRI...","Hingga Triwulan II 2026, total aset BRI Group ...","Hingga akhir Triwulan II 2026, Dana Pihak Keti..."
5,0,5068,Corporate,Kredit & Kualitas Aset,"kredit, pertumbuhan, tidak, bank, ekonomi, leb...","""BRI meyakini bahwa sinergi antara pemerintah ...",Dony menegaskan perubahan tata kelola BUMN dil...,Kebijakan tersebut mencerminkan sinergi fiskal...
6,24,590,Corporate,Kredit & Kualitas Aset,"kredit, tumbuh, pertumbuhan, pertumbuhan kredi...",Data dari Bank Indonesia (BI) menunjukkan bahw...,Pertumbuhan kredit diikuti dengan pertumbuhan ...,Pertumbuhan kredit ditopang oleh dana pihak ke...
7,37,310,Corporate,Kredit & Kualitas Aset,"npl, loan, rasio, rasio kredit, kredit bermasa...",Rasio kredit bermasalah atau non performing lo...,Rasio kredit bermasalah atau non-performing lo...,"Pada saat yang sama, kualitas aset terus memba..."
8,21,646,Corporate,Layanan & Produk,"digital, layanan, masyarakat, raya, inovasi, n...",Inovasi Produk Digital di Lini Bisnis Utama me...,Hal ini mencerminkan konsistensi pertumbuhan b...,Dukungan Bank Mandiri untuk Ekosistem Digital ...
9,22,625,Corporate,Layanan & Produk,"kartu, bni, atm, pembayaran, layanan, aplikasi...",Dia menyebut seluruh fitur BNI Mobile Banking ...,Kartu ATM BNI - Hubungi BNI Call 1500046 atau ...,Cara Tarik Tunai Tanpa Kartu di ATM BNI Buka a...


## Ringkasan: distribusi per aspek & per kategori

In [21]:
tot = info["Count"].sum()
print("Distribusi kalimat per ASPEK:")
for a, v in info.groupby("nama_aspek")["Count"].sum().sort_values(ascending=False).items():
    print(f"  {a:34} {v:>6,} ({v/tot*100:4.1f}%)")
print("\nDistribusi kalimat per KATEGORI FiQA:")
for k, v in info.groupby("kategori_fiqa")["Count"].sum().sort_values(ascending=False).items():
    print(f"  {k:12} {v:>6,} ({v/tot*100:4.1f}%)")

Distribusi kalimat per ASPEK:
  Lainnya                            21,577 (33.9%)
  Pergerakan IHSG & Indeks            7,108 (11.2%)
  Kredit & Kualitas Aset              5,968 ( 9.4%)
  Aksi Investor Asing                 4,267 ( 6.7%)
  Strategi & Ekspansi Bisnis          3,538 ( 5.6%)
  Sektor & Transaksi Pasar            3,335 ( 5.2%)
  Bursa Global                        3,089 ( 4.9%)
  Pergerakan Saham Bank               2,848 ( 4.5%)
  Dividen & Aksi Korporasi            2,496 ( 3.9%)
  Kebijakan Moneter & Nilai Tukar     2,494 ( 3.9%)
  Layanan & Produk                    1,860 ( 2.9%)
  Makroekonomi & Faktor Global        1,590 ( 2.5%)
  Kinerja Keuangan & Laba             1,470 ( 2.3%)
  Manajemen & Kepemimpinan              969 ( 1.5%)
  Saham Spesifik Non-Bank               723 ( 1.1%)
  Valuasi & Kapitalisasi                295 ( 0.5%)

Distribusi kalimat per KATEGORI FiQA:
  Lainnya      21,577 (33.9%)
  Corporate    16,301 (25.6%)
  Market       13,532 (21.3%)
  Stock  

## Terapkan ke tiap kalimat & simpan 2 output

In [22]:
sent["nama_aspek"]   = sent["topic"].map(aspek_of)
sent["kategori_fiqa"] = sent["nama_aspek"].map(kategori_of)

OUT = Path("data/processed")
mapping.to_csv(OUT / "mapping_aspek.csv", index=False, encoding="utf-8-sig")
sent.to_csv(OUT / "sentences_with_aspect.csv", index=False, encoding="utf-8-sig")
print(f"[1] mapping_aspek.csv -> {len(mapping)} topik, {info['nama_aspek'].nunique()-1} aspek + Lainnya")
print(f"[2] sentences_with_aspect.csv -> {len(sent):,} kalimat")

[1] mapping_aspek.csv -> 43 topik, 15 aspek + Lainnya
[2] sentences_with_aspect.csv -> 63,627 kalimat
